# FFAI Message Stack: What Gets Sent to the LLM

This notebook traces **exactly what messages** FFAI sends to the LLM API
across 10 turns, comparing two scenarios:

- **Scenario A**: Plain calls with `prompt_name` (no history references)
- **Scenario B**: Calls that reference prior turns via `history=[]` and `{{name.response}}`

Each turn prints the **actual `messages` array** that would be sent to
`litellm.completion()` — the system message, conversation history, and
user prompt with any injected `<conversation_history>` XML.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from unittest.mock import MagicMock, patch  # noqa: E402

from src.Clients.FFLiteLLMClient import FFLiteLLMClient  # noqa: E402
from src.FFAI import FFAI  # noqa: E402

In [ ]:
captured_messages = []
_turn_counter = [0]


def _fake_completion(**kwargs):
    _turn_counter[0] += 1
    messages = kwargs.get('messages', [])
    captured_messages.append({
        'turn': _turn_counter[0],
        'model': kwargs.get('model', ''),
        'messages': [dict(m) for m in messages],
    })

    mock_msg = MagicMock()
    mock_msg.content = f'Fake response for turn {_turn_counter[0]}.'
    mock_msg.tool_calls = None

    mock_choice = MagicMock()
    mock_choice.message = mock_msg

    mock_response = MagicMock()
    mock_response.choices = [mock_choice]
    mock_response.usage = MagicMock(
        prompt_tokens=10, completion_tokens=5, total_tokens=15,
    )
    return mock_response


def make_client():
    return FFLiteLLMClient(
        model_string='mistral/mistral-small-2503',
        api_key='fake-key',
        temperature=0.3,
        max_tokens=256,
        system_instructions='You are a helpful assistant.',
    )


def print_turn(turn_data, max_content=300):
    print(f'Turn {turn_data["turn"]} | model: {turn_data["model"]} | messages: {len(turn_data["messages"])}')
    print()
    for msg in turn_data['messages']:
        role = msg['role'].upper()
        content = msg['content']
        if len(content) > max_content:
            content = content[:max_content] + f'... [{len(msg["content"])} chars total]'
        print(f'  [{role}]')
        for line in content.split('\n'):
            print(f'    {line}')
        print()


PATCH_TARGET = 'src.Clients.FFLiteLLMClient.completion'
print('Capture utilities ready')

---

## Scenario A: 10 Plain Turns (No History References)

Each call uses `prompt_name` but does **not** reference any prior turns.
The client's internal `conversation_history` grows naturally — each turn
appends a user/assistant pair. The LLM sees **all prior turns** in the
messages array.

In [ ]:
captured_messages.clear()
_turn_counter[0] = 0

with patch(PATCH_TARGET, side_effect=_fake_completion):
    client_a = make_client()
    ffai_a = FFAI(client_a)

    for i in range(1, 11):
        ffai_a.generate_response(
            f'Turn {i}: Tell me a fact about the number {i}.',
            prompt_name=f'turn_{i}',
        )

print(f'Captured {len(captured_messages)} turns')
print(f'Client conversation_history: {len(client_a.conversation_history)} messages')

In [ ]:
# Show turns 1, 3, 5, 10 to see how the message stack grows
for turn_num in [1, 3, 5, 10]:
    print(f'========== SCENARIO A: Turn {turn_num} ==========')
    print_turn(captured_messages[turn_num - 1], max_content=120)
    print()

In [ ]:
print('Messages sent to API per turn (Scenario A):')
print()
for turn in captured_messages:
    n = len(turn['messages'])
    bar = '#' * n
    print(f'  Turn {turn["turn"]:2d}: {n:2d} messages  {bar}')

print()
print('Formula: 1 system + 2*(turn-1) prior conversation messages + 1 user = 2*turn - 1')

---

## Scenario B: 10 Turns With `history=[]` References

Each turn references the **previous turn** via `history=["prev_name"]`.
This triggers FFAI's **suspend/restore** mechanism:

1. FFAI saves the client's `conversation_history` and **clears** it
2. FFAI injects the referenced turn as `<conversation_history>` XML into the user message
3. The LLM sees only: system + 1 user message (with XML context inside)
4. After the call, FFAI **merges** old + new messages back into the client history

Result: the LLM only sees the **explicitly referenced** turn, not the full history.

In [ ]:
captured_messages.clear()
_turn_counter[0] = 0

with patch(PATCH_TARGET, side_effect=_fake_completion):
    client_b = make_client()
    ffai_b = FFAI(client_b)

    ffai_b.generate_response('My favorite color is blue.', prompt_name='turn_1')

    for i in range(2, 11):
        ffai_b.generate_response(
            f'Turn {i}: What was my favorite color? Just confirm it.',
            prompt_name=f'turn_{i}',
            history=[f'turn_{i-1}'],
        )

captured_b = list(captured_messages)
print(f'Captured {len(captured_b)} turns')
print(f'Client conversation_history: {len(client_b.conversation_history)} messages')

In [ ]:
# Show turns 1, 2, 3, 10 — notice the <conversation_history> XML
for turn_num in [1, 2, 3, 10]:
    history_ref = 'none' if turn_num == 1 else f'["turn_{turn_num-1}"]'
    print(f'========== SCENARIO B: Turn {turn_num} (history={history_ref}) ==========')
    print_turn(captured_b[turn_num - 1], max_content=500)
    print()

In [ ]:
# Side-by-side comparison
captured_messages.clear()
_turn_counter[0] = 0
with patch(PATCH_TARGET, side_effect=_fake_completion):
    client_a2 = make_client()
    ffai_a2 = FFAI(client_a2)
    for i in range(1, 11):
        ffai_a2.generate_response(f'Turn {i}: fact about {i}.', prompt_name=f'turn_{i}')
captured_a = list(captured_messages)

print(f'{"Turn":>4}  {"Plain (A)":>10}  {"History (B)":>11}')
print(f'{"----":>4}  {"---------":>10}  {"----------":>11}')
for i in range(10):
    a = len(captured_a[i]['messages'])
    b = len(captured_b[i]['messages'])
    print(f'{i+1:>4}  {a:>10}  {b:>11}')

print()
print('Plain (A):  messages grow linearly (client sends full conversation_history)')
print('History (B): messages stay at 2 (client history suspended, context injected as XML)')

---

## Scenario C: Interpolation with `{{name.response}}`

When a prompt contains `{{name.response}}`, FFAI **inline-substitutes** the
prior response directly into the prompt text. No `<conversation_history>` XML
is generated for interpolated names — they're deduplicated from the history block.

In [ ]:
captured_messages.clear()
_turn_counter[0] = 0

with patch(PATCH_TARGET, side_effect=_fake_completion):
    client_c = make_client()
    ffai_c = FFAI(client_c)

    ffai_c.generate_response(
        'Name one city in France and its population.',
        prompt_name='extract',
    )

    # Interpolation: {{extract.response}} is replaced with the actual response text
    ffai_c.generate_response(
        'Based on this fact: {{extract.response}} — Is this a large or small city?',
        prompt_name='evaluate',
    )

    # Both interpolation AND history — interpolated names are deduplicated from XML
    ffai_c.generate_response(
        'The fact was: {{extract.response}}. Do you agree: {{evaluate.response}}?',
        prompt_name='summarize',
        history=['extract', 'evaluate'],
    )

print(f'Captured {len(captured_messages)} turns')
print()
for turn in captured_messages:
    print(f'===== Turn {turn["turn"]} =====')
    print_turn(turn, max_content=250)
    print()

In [ ]:
# Key observation about Turn 3:
# extract and evaluate are interpolated inline, so they are NOT duplicated
# in <conversation_history> XML. Since ALL history names were interpolated,
# the XML block is empty and omitted entirely.

turn3_user = captured_messages[2]['messages'][-1]['content']
has_xml = '<conversation_history>' in turn3_user
print(f'Turn 3 user message contains <conversation_history> XML: {has_xml}')
print(f'Turn 3 user message length: {len(turn3_user)} chars')
print()
print('The deduplication rule: if a name appears in both {{name.response}}')
print('and history=[...], it is interpolated inline and removed from the XML block.')

---

## Complete 10-Turn Summary Table

The table below shows every turn across all 3 scenarios. For each turn it shows:

- **Full stack sent?** — Whether the client's growing `conversation_history` is included in the API messages
- **What else is sent** — Any additional context beyond the current prompt (XML blocks, interpolated values)

In [ ]:
# Run all 3 scenarios and build a comparison table

results = {}

# --- Scenario A: Plain ---
captured_messages.clear()
_turn_counter[0] = 0
with patch(PATCH_TARGET, side_effect=_fake_completion):
    c = make_client()
    f = FFAI(c)
    for i in range(1, 11):
        f.generate_response(f'Turn {i}: fact about {i}.', prompt_name=f'turn_{i}')
results['A'] = list(captured_messages)

# --- Scenario B: history=[prev] ---
captured_messages.clear()
_turn_counter[0] = 0
with patch(PATCH_TARGET, side_effect=_fake_completion):
    c = make_client()
    f = FFAI(c)
    f.generate_response('My favorite color is blue.', prompt_name='turn_1')
    for i in range(2, 11):
        f.generate_response(
            f'Turn {i}: confirm my favorite color.',
            prompt_name=f'turn_{i}',
            history=[f'turn_{i-1}'],
        )
results['B'] = list(captured_messages)

# --- Scenario C: interpolation ---
captured_messages.clear()
_turn_counter[0] = 0
with patch(PATCH_TARGET, side_effect=_fake_completion):
    c = make_client()
    f = FFAI(c)
    f.generate_response('My favorite color is blue.', prompt_name='turn_1')
    for i in range(2, 11):
        f.generate_response(
            f'You said: {{{{turn_{i-1}.response}}}} What color is that?',
            prompt_name=f'turn_{i}',
        )
results['C'] = list(captured_messages)


def describe_extra(turn_data):
    """Describe what extra context is in the user message beyond the prompt."""
    msgs = turn_data['messages']
    user_content = msgs[-1]['content']

    extras = []
    if '<conversation_history>' in user_content:
        # Count how many <interaction> blocks
        count = user_content.count('<interaction ')
        extras.append(f'{count} prior turn(s) as XML')

    # Check for inline interpolation (Fake response text appearing)
    if 'Fake response' in user_content and '<conversation_history>' not in user_content:
        count = user_content.count('Fake response')
        extras.append(f'{count} interpolated value(s)')

    return ', '.join(extras) if extras else 'None'


def full_stack_sent(turn_data):
    n = len(turn_data['messages'])
    return n > 2


print(f'{"":>3} {"--- Scenario A: Plain ---":^32} {"--- Scenario B: history=[] ---":^36} {"--- Scenario C: {{}} interpolation ---":^40}')
print(f'{"#":>3} {"Full":^7} {"Msgs":>4}  {"Extra context":^18}  {"Full":^7} {"Msgs":>4}  {"Extra context":^22}  {"Full":^7} {"Msgs":>4}  {"Extra context":^26}')
print(f'{"":>3} {"stack?":^7} {"":>4}  {"":^18}  {"stack?":^7} {"":>4}  {"":^22}  {"stack?":^7} {"":>4}  {"":^26}')
print(f'{"---":>3} {"-------":>7} {"----":>4}  {"------------------":>18}  {"-------":>7} {"----":>4}  {"----------------------":>22}  {"-------":>7} {"----":>4}  {"--------------------------":>26}')

for i in range(10):
    a = results['A'][i]
    b = results['B'][i]
    c_data = results['C'][i]

    a_stack = 'Yes' if full_stack_sent(a) else 'No'
    b_stack = 'Yes' if full_stack_sent(b) else 'No'
    c_stack = 'Yes' if full_stack_sent(c_data) else 'No'

    a_extra = describe_extra(a)
    b_extra = describe_extra(b)
    c_extra = describe_extra(c_data)

    print(f'{i+1:>3}  {a_stack:^7} {len(a["messages"]):>4}  {a_extra:>18}  {b_stack:^7} {len(b["messages"]):>4}  {b_extra:>22}  {c_stack:^7} {len(c_data["messages"]):>4}  {c_extra:>26}')

print()
print('Full stack = client conversation_history included in API messages (grows each turn)')
print('No full stack = client history suspended; only current prompt (with context injected into it)')

---

## Key Takeaways

| Mechanism | Trigger | What the LLM sees |
|-----------|---------|-------------------|
| **Plain call** | No `history`, no `{{}}` | All prior turns in `messages` array. Grows as `2n-1`. |
| **`history=["X"]`** | Explicit history list | Only turn X as `<conversation_history>` XML in user message. Always 2 messages. |
| **`{{X.response}}`** | Interpolation pattern | Response of X **inline** in the prompt. No XML. Always 2 messages. |
| **Both `history` + `{{}}`** | Mixed | Interpolated names are **deduplicated** from XML. Only non-interpolated names appear in XML. |

**How suspend/restore works:**

1. Before a call with `history` or `{{}}`, FFAI **saves** the client's `conversation_history` and **clears** it
2. The LLM sees only `system + user` (no growing history)
3. After the call, FFAI **merges** saved + new messages back into the client
4. The client's own history keeps growing internally, but the LLM never sees the full thing

**Message growth by scenario:**

- **Plain**: `2n - 1` messages (system + accumulated user/assistant pairs + current user)
- **History/Interpolation**: Always `2` messages (system + user with context embedded)

The `history` and interpolation mechanisms give you **explicit control** over what
context the LLM sees, keeping token usage predictable regardless of session length.